# GPT-2 small activation atlas: exploring stage 1

Reads the outputs of `python -m atlas` (see `docs/atlas-design.md`) and looks at what they say about GPT-2 small.
Build them first with `run-atlas` (1M OpenWebText tokens, about 4 minutes on an M4).

1. Residual-stream geometry: how variance and effective dimension grow with depth
2. MLP neurons: firing rates and peak activations by layer
3. A single neuron: summary, activation histogram and top contexts
4. Regression check: stored exemplar pointers reproduce on recompute

All statistics exclude position 0 (BOS) and are conditional on OpenWebText and positions 1–127.

In [ ]:
import json
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import torch
from atlas import HFGPT2, hook_layer, load_tokenizer

ATLAS = Path("../results/atlas")
manifest = json.loads((ATLAS / "manifest.json").read_text())
neurons = duckdb.sql(f"select * from '{ATLAS / 'neurons.parquet'}'").df()
topk = duckdb.sql(f"select * from '{ATLAS / 'topk.parquet'}'").df()
resid = np.load(ATLAS / "resid.npz")
hists = np.load(ATLAS / "histograms.npz")
tokens = np.load(ATLAS / "tokens.npy", mmap_mode="r")

# Categorical slots 1-4 and text inks of the dataviz default palette, as in ioi_hydra.ipynb.
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e5e4df"
def tidy(ax):
    ax.grid(axis="y", color=GRID, zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

print(f"{manifest['n_rows']:,} rows x {manifest['seq_len']} tokens from {manifest['n_docs']} docs ({manifest['dataset']} shard {manifest['shard']})")
print(f"{neurons['n'].iloc[0]:,} tokens per unit; {len(neurons):,} units; {len(topk):,} top-k exemplars")

## 1. Residual-stream geometry

From the per-layer covariance in `resid.npz` ("layer 12" is the final residual stream, the input to `ln_final`).

- **Total variance**, the trace of the covariance: how much the stream's spread grows with depth.
- **Effective dimension**, the participation ratio $(\sum_i \lambda_i)^2 / \sum_i \lambda_i^2$ of the eigenvalues: about
  1 if one direction carries everything, 768 if variance is spread evenly.
- **Largest single basis dimension's share of variance**: the residual basis is not privileged in general, but GPT-2 is
  known to have a few outlier dimensions (447 among them).

*Expect:* variance growing roughly exponentially with depth, and a low effective dimension where outlier dimensions dominate.

In [ ]:
layers, cov = resid["layer"], resid["cov"]
eig = np.linalg.eigvalsh(cov)[:, ::-1]  # [13, 768], descending
total = eig.sum(1)
pr = total**2 / (eig**2).sum(1)
diag = np.diagonal(cov, axis1=1, axis2=2)
top_dim, top_share = diag.argmax(1), diag.max(1) / total
dims90 = (np.cumsum(eig, 1) / total[:, None] < 0.9).sum(1) + 1

print("layer  total var  eff. dim  dims for 90%  top basis dim (share)")
for l in layers:
    print(f"{l:5d}  {total[l]:9.1f}  {pr[l]:8.1f}  {dims90[l]:12d}  {top_dim[l]:4d} ({top_share[l]:.0%})")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(layers, total, "o-", color=C[0], lw=2, ms=5)
axes[0].set(yscale="log", title="Total variance grows with depth", xlabel="layer (12 = final residual)", ylabel="trace of covariance")
axes[1].plot(layers, pr, "o-", color=C[0], lw=2, ms=5)
axes[1].set(title="Effective dimension (participation ratio)", xlabel="layer (12 = final residual)", ylabel="dimensions", ylim=(0, None))
for i, l in enumerate([0, 4, 8, 12]):
    axes[2].plot(np.arange(1, 769), np.cumsum(eig[l]) / total[l], color=C[i], lw=2, label=f"layer {l}")
axes[2].set(xscale="log", title="Cumulative variance explained by top PCs", xlabel="number of principal components", ylabel="fraction of variance", ylim=(0, 1.02))
axes[2].legend(frameon=False, loc="lower right")
for ax in axes:
    tidy(ax)
plt.tight_layout()

**Reading it.** The first block multiplies the stream's variance by about 90 (layer 0 is the embedding alone), and it
grows steadily after that, another 3× in the last block. The effective dimension rises from 14 to about 170 by layers
7–9, then collapses to about 7 at the final residual stream, where a handful of directions carry most of the variance
(the top principal component alone carries about a third). The largest-variance basis dimension is 480 in layers 1–7,
447 in layers 8–10 and 496 in layers 11–12.

## 2. MLP neurons

Each neuron's firing rate (fraction of tokens with post-GELU activation > 0) and its peak activation, summarized per
layer as the median with a 10th–90th percentile band. Rarely-firing neurons are those with a firing rate below 1e-4,
fewer than about 100 tokens in 1M.

*Expect:* early layers to hold most of the rarely-firing neurons, and peak activations to grow in the last layers.

In [ ]:
mlp = neurons[neurons.kind == "post"]
by = mlp.groupby("layer")
q = lambda col, p: by[col].quantile(p).to_numpy()
L = np.arange(12)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col, title, ylabel, log in [
    (axes[0], "frac_pos", "Firing rate per neuron", "tokens with activation > 0", True),
    (axes[1], "max", "Peak activation per neuron", "max activation", False),
]:
    ax.fill_between(L, q(col, 0.1), q(col, 0.9), color=C[0], alpha=0.15, lw=0, label="10th-90th percentile")
    ax.plot(L, q(col, 0.5), "o-", color=C[0], lw=2, ms=5, label="median")
    ax.set(title=title, xlabel="layer", ylabel=ylabel, yscale="log" if log else "linear")
    if log:
        ax.set_yticks([0.01, 0.02, 0.05, 0.1, 0.2, 0.5], ["1%", "2%", "5%", "10%", "20%", "50%"])
        ax.yaxis.set_minor_locator(plt.matplotlib.ticker.NullLocator())
    ax.legend(frameon=False)
rare = by["frac_pos"].apply(lambda s: (s < 1e-4).sum()).to_numpy()
axes[2].bar(L, rare, color=C[0], width=0.6, zorder=2)
for l, n in zip(L, rare):
    if n:
        axes[2].text(l, n, str(n), ha="center", va="bottom", fontsize=9, color=INK2)
axes[2].set(title="Rarely-firing neurons (rate < 1e-4)", xlabel="layer", ylabel="neurons (of 3072)")
for ax in axes:
    ax.set_xticks(L)
    tidy(ax)
plt.tight_layout()

print("highest-peak neurons:")
print(mlp.nlargest(5, "max")[["layer", "unit", "max", "mean", "std", "frac_pos", "p999"]].to_string(index=False))

**Reading it.** Layers 0–2 hold most of the rarely-firing neurons (63, 86 and 21 of 3072). Layer 1 is the odd one out:
its median neuron fires on about 30% of tokens, more than twice the median of any other layer. Peak activations climb steadily
through the last layers, and the highest-peak neurons are all in layers 10–11.

## 3. A single neuron

`show_neuron(layer, unit)` prints the neuron's summary row, plots its activation histogram (from `histograms.npz`, on
an asinh axis: linear near 0, logarithmic in the tails) and lists its top-activating contexts with the token marked «like this».

In [ ]:
EDGES = hists["edges"]

def show_neuron(layer, unit, n=10, hook=None):
    hook = hook or f"blocks.{layer}.mlp.hook_post"
    row = neurons[(neurons.hook == hook) & (neurons.unit == unit)].iloc[0]
    print(f"{hook} unit {unit}: mean {row['mean']:.3f}, std {row['std']:.3f}, max {row['max']:.2f}, "
          f"fires on {row.frac_pos:.1%} of tokens, p50 {row.p50:.3f}, p99 {row.p99:.2f}, p99.9 {row.p999:.2f}")
    counts = hists[hook][unit]
    fig, ax = plt.subplots(figsize=(7, 2.6))
    ax.stairs(counts[1:-1], EDGES, color=C[0], lw=1.5, fill=True, alpha=0.3)
    ax.stairs(counts[1:-1], EDGES, color=C[0], lw=1.5)
    ax.axvline(row["max"], color=INK2, lw=1, ls="--")
    ax.annotate("max", (row["max"], 0.9), xycoords=("data", "axes fraction"), xytext=(4, 0), textcoords="offset points", fontsize=9, color=INK2)
    nz = np.nonzero(counts[1:-1])[0]
    lo, hi = EDGES[nz[0]], EDGES[nz[-1] + 1]
    ax.set_xscale("asinh", linear_width=0.05)
    ticks = [t for t in (-1, -0.1, 0, 0.1, 0.3, 1, 3, 10, 30, 100) if lo <= t <= hi]
    ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
    ax.xaxis.set_minor_locator(plt.matplotlib.ticker.NullLocator())
    ax.set(yscale="log", xlim=(lo, hi),
           title=f"layer {layer} neuron {unit}: activation histogram (subsampled tokens)", xlabel="activation", ylabel="tokens")
    tidy(ax)
    plt.show()
    t = topk[(topk.hook == hook) & (topk.unit == unit)].sort_values("rank").head(n)
    for v, ctx in zip(t.value, t.context):
        print(f"  {v:6.2f}  {ctx[-90:]}")

show_neuron(9, 1000)

In [ ]:
# The neuron with the largest peak activation in the last layer.
l11 = mlp[mlp.layer == 11].nlargest(1, "max").iloc[0]
show_neuron(11, int(l11.unit))

## 4. Regression check: pointers reproduce

Activations are a pure function of (weights, tokens), so the atlas stores exemplars as `(row, pos)` pointers. Recompute
a random sample of top-k entries and check that each stored value comes back.

In [ ]:
model = HFGPT2("cpu")
sample = topk.sample(12, random_state=0)
worst = 0.0
for r in sample.itertuples():
    got = {}
    toks = torch.tensor(tokens[[r.row]].astype(np.int64))
    model.run(toks, {r.hook: lambda act: got.__setitem__("x", act[0, r.pos, r.unit].item())})
    worst = max(worst, abs(got["x"] - r.value) / max(abs(r.value), 1e-3))
    print(f"{r.hook:26s} unit {r.unit:4d} rank {r.rank:2d}: stored {r.value:9.4f}  recomputed {got['x']:9.4f}")
print(f"worst relative difference {worst:.1e}")
assert worst < 1e-3